In [15]:
import numpy as np
from scipy.optimize import least_squares
import matplotlib.pyplot as plt

In [16]:
### define forward model

def model_Qb(t_obs, Q, r, S, T, t0):
    """
    Forward model of boundary flux:
        t_obs: observed elapsed times since t0 [T]
        Q: pumping rate (positive for extraction) [L^3/T]
        r: radial distance(s) [L] (scalar or array)
        S: storage coefficient [-]
        t0: initial time
    
    Notes:
        - If any (t_obs - t0) <= 0, returns tiny values to keep residuals large

    """
    t_obs = np.asarray(t_obs, dtype=float)
    tau = t_obs - t0
    tiny = 1e-300
    if np.any(tau <= 0):
        return tiny * np.ones_like(t_obs)
    u = (r**2 * S) / (4.0 * T * tau)
    return Q * np.exp(-u)

In [17]:
### calculate analytic residuals and Jacobian matrix

def residuals_and_jacobian(params, t_obs, Qb_obs, Q, S, T, eps_floor=1e-12):
    """
    Calculate (residuals, J) where:
        - residuals: res_i = ln(Qb_obs) - ln(Qb_pred)  (length N)
        - J: N x 2 Jacobian with columns [d res / d r, d res / d t0]
        - params = [r, t0]

    """
    r, t0 = params
    t_obs = np.asarray(t_obs, dtype=float)
    Qb_obs = np.asarray(Qb_obs, dtype=float)

    # bounds: r <= 0, tau <= 0
    if r <= 0:
        N = len(t_obs)
        res = 1e6 * np.ones(N)
        J = np.zeros((N, 2))
        return res, J
    tau = t_obs - t0
    if np.any(tau <= 0):
        N = len(t_obs)
        res = 1e6 * np.ones(N)
        J = np.zeros((N, 2))
        return res, J

    # model prediction
    Qb_pred = model_Qb(t_obs, Q, r, S, T, t0)
    Qb_obs_safe = np.maximum(Qb_obs, eps_floor)   # floor values for log safety
    Qb_pred_safe = np.maximum(Qb_pred, eps_floor) # floor values for log safety

    # residuals in log-space
    res = np.log(Qb_obs_safe) - np.log(Qb_pred_safe)

    # Jacobian matrix entries
    dres_dr = (r * S) / (2.0 * T * tau)
    dres_dt0 = (r**2 * S) / (4.0 * T * (tau**2))

    J = np.vstack((dres_dr, dres_dt0)).T  # shape (N, 2)
    return res, J

In [18]:
### define least-squares wrapper functions

def fun_for_ls(params, t_obs, Qb_obs, Q, S, T):
    res, _ = residuals_and_jacobian(params, t_obs, Qb_obs, Q, S, T)
    return res

def jac_for_ls(params, t_obs, Qb_obs, Q, S, T):
    _, J = residuals_and_jacobian(params, t_obs, Qb_obs, Q, S, T)
    return J

In [19]:
### generate synthetic data

# "true" synthetic parameters
Q_true = 1.0e-3      # m^3/s
r_true = 2.5         # m
S = 1e-4             # dimensionless (known)
T = 1e-3             # m^2/s (known)
t0_true = 0.0        # seconds (pump started at time 0)

# observation times (seconds) - choose spread so tau varies
t_obs_full = np.array([10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0, 10000.0])

# generate clean boundary flux
Qb_clean_full = model_Qb(t_obs_full, Q_true, r_true, S, T, t0_true)

#add noise to boundary flux
rng = np.random.default_rng(42)
rel_noise = 0.03  # 3% relative Gaussian noise
Qb_obs_full = Qb_clean_full * (1.0 + rel_noise * rng.standard_normal(len(Qb_clean_full)))

# initial guesses 
# bounds: r > 0, t0 < min(t_obs)
r0 = 1.0
t0_guess = -5.0
lower_bounds = [1e-9, -1e9]  
upper_bounds = [1e6, np.min(t_obs) - 1e-9]

In [20]:
### run least-squares with analytic Jacobian

est_r = []
est_t0 = []
se_r = []
se_t0 = []
n_points = []
latest_t = []

# iterate adding observations in chronological order, calculate covariance, SEs, correlation, diagnostics
for k in range(1, len(t_obs_full)+1):
    t_obs = t_obs_full[:k]
    Qb_obs = Qb_obs_full[:k]

    x0 = np.array([r0, t0_guess])
    res = least_squares(fun_for_ls, x0, jac=jac_for_ls,
                        bounds=(lower_bounds, upper_bounds),
                        args=(t_obs, Qb_obs, Q_true, S, T),
                        method='trf', xtol=1e-12, ftol=1e-12)

    theta_hat = res.x
    r_hat, t0_hat = theta_hat

    residuals, J = residuals_and_jacobian(theta_hat, t_obs, Qb_obs, Q_true, S, T)
    N = residuals.size
    p = 2
    RSS = np.sum(residuals**2)
    sigma2_hat = RSS / max(1, N - p)

    JTJ = J.T @ J
    cond_JTJ = np.linalg.cond(JTJ)

    # robust inversion for covariance
    if cond_JTJ < 1e12:
        inv_JTJ = np.linalg.inv(JTJ)
    else:
        inv_JTJ = np.linalg.pinv(JTJ)

    cov_approx = sigma2_hat * inv_JTJ
    se = np.sqrt(np.abs(np.diag(cov_approx)))  # abs to avoid tiny negative numerical

    # save results
    est_r.append(r_hat)
    est_t0.append(t0_hat)
    se_r.append(se[0])
    se_t0.append(se[1])
    n_points.append(k)
    latest_t.append(t_obs[-1])

    # update initial guess for next iteration
    r0 = r_hat
    t0_guess = t0_hat

In [ ]:
### evaluate identifiability

# get residuals and Jacobian at the fitted parameters
residuals, J = residuals_and_jacobian(theta_hat, t_obs, Qb_obs, Q_true, S, T)

# calculate JTJ and condition number
JTJ = J.T @ J
cond = np.linalg.cond(JTJ)
print("cond(J^T J) =", round(cond,0))

# Small (≈ 1): well-conditioned -- all parameter directions are similarly constrained by the data
# Moderate (10–1e6): still usually OK -- inversion and covariance estimation are numerically stable
# Large (≫ 1e12, machine-dependent): ill-conditioned -- numerical inversion of J^T J is unstable, and the estimated covariance (and SEs) may be meaningless

cond(J^T J) = 5513.0


In [27]:
# get eigenvalues of J^T J
w, _ = np.linalg.eig(JTJ)
print("eigenvalues:", np.sort(w)[::-1])

# if one eigenvalue is near zero (or many are tiny), it points to one or more nearly-unidentifiable parameter combos

# calculate parameter correlation from covariance
N = residuals.size
p = theta_hat.size
RSS = np.sum(residuals**2)
sigma2_hat = RSS / max(1, N - p)
print("RSS:", RSS, " sigma2_hat:", sigma2_hat)
cov = sigma2_hat * np.linalg.pinv(JTJ)
se = np.sqrt(np.diag(cov))
corr = cov / np.outer(se, se)
print("corr matrix:\n", corr)

# Small SE relative to parameter value: precise estimate
# Large SE (comparable to or larger than parameter): poorly constrained parameter

eigenvalues: [6.01415585e-05 1.09083238e-08]
RSS: 0.007462034568118904  sigma2_hat: 0.0014924069136237807
corr matrix:
 [[ 1.         -0.96886076]
 [-0.96886076  1.        ]]
